In [0]:
spark.sql("USE workspace.ventes")

spark.sql("""
CREATE OR REPLACE TABLE gold_ca_jour AS
SELECT
    date,
    magasin,
    COUNT(*)                          AS nb_ventes,
    SUM(quantite)                     AS nb_articles,
    ROUND(SUM(chiffre_affaires), 2)   AS ca_total
FROM silver_ventes
GROUP BY date, magasin
""")

display(spark.sql("SELECT * FROM gold_ca_jour ORDER BY date, magasin"))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_top_produits AS
SELECT
    produit,
    SUM(quantite)                     AS quantite_vendue,
    ROUND(SUM(chiffre_affaires), 2)   AS ca_total,
    RANK() OVER (ORDER BY SUM(chiffre_affaires) DESC) AS rang
FROM silver_ventes
GROUP BY produit
""")

display(spark.sql("SELECT * FROM gold_top_produits ORDER BY rang"))

In [0]:
display(spark.sql("""
SELECT
    date,
    magasin,
    ca_total,
    ROUND(ca_total - LAG(ca_total) OVER (PARTITION BY magasin ORDER BY date), 2) AS variation_veille,
    ROUND(SUM(ca_total) OVER (PARTITION BY magasin ORDER BY date), 2)            AS ca_cumule
FROM gold_ca_jour
ORDER BY magasin, date
"""))

## Couche Gold : agrégats des ventes

- `gold_ca_jour` : chiffre d'affaires par magasin et par jour
- `gold_top_produits` : classement des produits par chiffre d'affaires
- Analyse : variation par rapport à la veille et chiffre d'affaires cumulé par magasin